# Figure 4c - AF3 (31 - PTI-PAE) vs AF-TCRdock

Per-antigen / per-TCR AUCs for the two classifiers, arranged as **four AF3 vs
AF-TCRdock pairs** (MHC class x orientation), with connecting lines between matched
triads and paired Wilcoxon signed-rank p-values.

| set | AF3 n | AF-TCRdock n | AF3 median | TCRdock median | Wilcoxon p |
|---|---|---|---|---|---|
| I, Ag-centric  | 14 | 12 | 0.92 | 0.59 | 0.006 |
| II, Ag-centric | 8  | 8  | 0.81 | 0.59 | 0.008 |
| I, TCR-centric | 14 | 12 | 0.86 | 0.59 | 0.006 |
| II, TCR-centric| 205| 205| 0.94 | 0.70 | 1.6e-08 |

AF3 n > AF-TCRdock n for class I because AF-TCRdock cannot model the two
cross-species KRAS antigens. Connecting lines join matched triads (12 / 8 / ~10 /
205); class I TCR-centric matches AF3-per-TCR AUCs to AF-TCRdock per their antigen.
Data: `antigen_centric.csv`, `fig4c_tcrdock_per_antigen.csv`,
`classII_per_TCR_AUC.csv`, `fig4c_tcrdock_summary.csv`, `../supplementaryTable3_updated.xlsx`.


In [1]:
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score

CI_BLUE="#3a78b8"; CII_RED="#c83737"; LIGHT_BLUE="#9bbedf"; LIGHT_RED="#e89c9c"; MEDRED="#b81d1d"

def mkey(cls, peptide, mhc2):
    return peptide if cls=='I' else f"{peptide}|{mhc2}"

# ---- AF3 antigen-centric ----
af3_ant = pd.read_csv('antigen_centric.csv')
af3_ant['k'] = [mkey(c,p,m) for c,p,m in zip(af3_ant['mhc_class'],af3_ant['peptide'],af3_ant['mhc_2_name'])]

# ---- AF3 TCR-centric (recompute from supp table 3) ----
st3 = pd.read_excel('../supplementaryTable3_updated.xlsx')
st3['PTI_PAE'] = 31 - st3['mean_p_tcr_interface_pae']
st3['antigen'] = np.where(st3['mhc_class']=='I',
    st3['mhc_1_name'].astype(str)+':'+st3['peptide'],
    st3['mhc_1_name'].astype(str)+','+st3['mhc_2_name'].astype(str)+':'+st3['peptide'])
st3['tcr_pair'] = st3['tcr_1_seq'].astype(str)+'|'+st3['tcr_2_seq'].astype(str)
st3['k'] = [mkey(c,p,m) for c,p,m in zip(st3['mhc_class'],st3['peptide'],st3['mhc_2_name'])]
rows=[]
for mc in ['I','II']:
    for ag,g in st3[st3['mhc_class']==mc].groupby('antigen'):
        cog=g[g['cognate']==True]; nc=g[g['cognate']==False]
        if len(nc)==0: continue
        ncs=nc['PTI_PAE'].values
        for _,c in cog.iterrows():
            yt=np.concatenate([[1],np.zeros(len(ncs),int)]); ys=np.concatenate([[c['PTI_PAE']],ncs])
            try: a=roc_auc_score(yt,ys)
            except: continue
            rows.append({'mhc_class':mc,'k':c['k'],'tcr_pair':c['tcr_pair'],'auc':a})
per_pair=pd.DataFrame(rows)
af3_tcr=per_pair.groupby(['mhc_class','tcr_pair'],as_index=False).agg(auc=('auc','median'),k=('k','first'))

# ---- TCRdock ----
tdk_pa = pd.read_csv('fig4c_tcrdock_per_antigen.csv')
tdk_pa['peptide']=tdk_pa['antigen'].str.split(':').str[-1]
tdk_pa['mhc2']=tdk_pa['antigen'].str.rsplit(':',n=1).str[0].str.split(',').str[-1]
tdk_pa['k']=[mkey(c,p,m) for c,p,m in zip(tdk_pa['mhc_class'],tdk_pa['peptide'],tdk_pa['mhc2'])]
tdk_II = pd.read_csv('classII_per_TCR_AUC.csv')
summ = pd.read_csv('fig4c_tcrdock_summary.csv')
def wp(mc,o): return summ[(summ['mhc_class']==mc)&(summ['orientation']==o)]['paired_wilcoxon_p'].iloc[0]

def build(af3_df, cls, orient):
    a3 = af3_df[af3_df['mhc_class']==cls]
    dk = tdk_pa[tdk_pa['mhc_class']==cls]
    m = a3[['k','auc']].merge(dk[['k','TCRdock_AUC']], on='k', how='inner')
    return dict(cls=cls, af3=a3['auc'].values, dock=dk['TCRdock_AUC'].values,
                pairs=list(zip(m['auc'],m['TCRdock_AUC'])), p=wp(cls,orient))

sets=[ build(af3_ant,'I','antigen-centric'), build(af3_ant,'II','antigen-centric'),
       build(af3_tcr,'I','TCR-centric') ]
sets.append(dict(cls='II', af3=tdk_II['AUC_AF3_PTI_PAE'].values, dock=tdk_II['AUC_TCRdock_pmhc_tcr_pae'].values,
                 pairs=list(zip(tdk_II['AUC_AF3_PTI_PAE'],tdk_II['AUC_TCRdock_pmhc_tcr_pae'])), p=wp('II','TCR-centric')))
labels=['I\n(Ag)','II\n(Ag)','I\n(TCR)','II\n(TCR)']
for lab,s in zip(labels,sets):
    print(f"{lab.split(chr(10))}: AF3 n={len(s['af3'])} med={np.median(s['af3']):.3f} | dock n={len(s['dock'])} med={np.median(s['dock']):.3f} | pairs={len(s['pairs'])}")

# ===================== PLOT =====================
fig, ax = plt.subplots(figsize=(12.5, 7.2))
gap=3.0; pair_off=0.62
centers=[i*gap for i in range(4)]
rng=np.random.default_rng(0)

def full_violin(ax, x, vals, color):
    if len(vals)<2: return
    from scipy.stats import gaussian_kde
    vals=np.asarray(vals); kde=gaussian_kde(vals)
    ys=np.linspace(max(0,vals.min()-0.05), min(1.02,vals.max()+0.05),120)
    dens=kde(ys); dens=dens/dens.max()*0.42  # half-width per side (full width = 0.84)
    ax.fill_betweenx(ys, x - dens, x + dens, color=color, alpha=0.45, lw=1.0,
                     edgecolor=color, zorder=2)

for ci,(c,s) in enumerate(zip(centers,sets)):
    col = CI_BLUE if s['cls']=='I' else CII_RED
    xa, xd = c-pair_off, c+pair_off
    full_violin(ax, xa, s['af3'], col)
    full_violin(ax, xd, s['dock'], col)
    # connecting lines (matched)
    for a,d in s['pairs']:
        ax.plot([xa,xd],[a,d], color='0.5', lw=0.5, alpha=0.35, zorder=1)
    # scatter
    jx=0.10
    ax.scatter(xa+rng.uniform(-jx,jx,len(s['af3'])), s['af3'], s=22, color=col,
               edgecolors='k', linewidths=0.3, alpha=0.8, zorder=4)
    ax.scatter(xd+rng.uniform(-jx,jx,len(s['dock'])), s['dock'], s=22, color=col,
               edgecolors='k', linewidths=0.3, alpha=0.4, zorder=4)
    # medians
    ax.plot([xa-0.42,xa+0.42],[np.median(s['af3'])]*2, color=MEDRED, lw=3, zorder=6, solid_capstyle='round')
    ax.plot([xd-0.42,xd+0.42],[np.median(s['dock'])]*2, color=MEDRED, lw=3, zorder=6, solid_capstyle='round')
    ax.text(xa, 1.13, f"{np.median(s['af3']):.2f}", ha='center', fontsize=15, fontweight='bold', color=col)
    ax.text(xd, 1.13, f"{np.median(s['dock']):.2f}", ha='center', fontsize=15, fontweight='bold', color='0.45')
    ax.text(xa, 1.07, f"n={len(s['af3'])}", ha='center', fontsize=13, color='0.4')
    ax.text(xd, 1.07, f"n={len(s['dock'])}", ha='center', fontsize=13, color='0.4')
    # p-value bracket over pair
    pstr = f"p={s['p']:.1e}" if s['p']<1e-3 else f"p={s['p']:.3f}"
    ytop=1.20
    ax.plot([xa,xa,xd,xd],[ytop-0.012,ytop,ytop,ytop-0.012], color='0.3', lw=1.0)
    ax.text(c, ytop+0.006, pstr, ha='center', fontsize=13)

ax.axhline(0.5, ls='--', color='0.6', lw=1, zorder=0)
ax.set_ylim(-0.03,1.30); ax.set_xlim(centers[0]-1.4, centers[-1]+1.4)
ax.set_ylabel('Per-antigen / per-TCR AUC', fontsize=17)
ax.set_yticks([0,0.2,0.4,0.6,0.8,1.0]); ax.tick_params(axis='y', labelsize=15)
ax.set_xticks([]); ax.spines[['top','right']].set_visible(False)

# ---- structured x-axis table ----
def row(y, label):
    ax.annotate(label, xy=(0,0), xytext=(centers[0]-1.55, y), textcoords=('data'),
                ha='right', va='center', fontsize=13.5, fontweight='bold', annotation_clip=False)
y0=-0.10; dy=0.085
for ci,c in enumerate(centers):
    xa,xd=c-pair_off,c+pair_off
    ax.text(xa, y0, 'AF3', ha='center', va='center', fontsize=12.5, clip_on=False)
    ax.text(xd, y0, 'AF-\nTCRdock', ha='center', va='center', fontsize=11, color='0.4', clip_on=False)
    ax.text(c, y0-dy, 'I' if sets[ci]['cls']=='I' else 'II', ha='center', va='center', fontsize=14, clip_on=False)
# orientation spans
ax.text((centers[0]+centers[1])/2, y0-2*dy, 'Ag-centric', ha='center', va='center', fontsize=13, clip_on=False)
ax.text((centers[2]+centers[3])/2, y0-2*dy, 'TCR-centric', ha='center', va='center', fontsize=13, clip_on=False)
row(y0,'model:'); row(y0-dy,'MHC class:'); row(y0-2*dy,'orientation:')
# light separators under orientation groups
for (a,b) in [(centers[0]-pair_off-0.5,centers[1]+pair_off+0.5),(centers[2]-pair_off-0.5,centers[3]+pair_off+0.5)]:
    ax.plot([a,b],[y0-2*dy+0.035]*2, color='0.7', lw=0.8, clip_on=False)

ax.set_title('Figure 4c \u2014 AF3 PTI-PAE vs AF-TCRdock across MHC class \u00d7 orientation', fontsize=16, pad=28)
fig.subplots_adjust(bottom=0.26, top=0.86)
fig.savefig('Fig4_c_AF3_vs_AFTCRdock.png', dpi=300, bbox_inches='tight')
print('saved')


['I', '(Ag)']: AF3 n=14 med=0.920 | dock n=12 med=0.591 | pairs=12
['II', '(Ag)']: AF3 n=8 med=0.810 | dock n=8 med=0.592 | pairs=8
['I', '(TCR)']: AF3 n=14 med=0.858 | dock n=12 med=0.591 | pairs=10
['II', '(TCR)']: AF3 n=205 med=0.937 | dock n=205 med=0.703 | pairs=205


saved
